# 02. EDA e Validação das Regras de Negócio

**Input:** `data/cleaned_telemetry.parquet`, `data/Alarmes - Regra de Negocio.xlsx`, `data/desenvolver_dontgo.xlsx`  
**Output:** `data/business_rules_validation.csv`, gráficos de EDA

Objetivos:
- Entender a distribuição dos alarmes e a taxa de Don't Go
- Reproduzir o sinal `Is_Dont_Go` via janela deslizante
- Documentar falsos positivos e falsos negativos

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils import set_seeds
from src.business_rules import load_rules, apply_sliding_window, compare_signals

set_seeds(42)
sns.set_theme(style='whitegrid')

DATA_DIR = '../data'

## 1. Carregamento

In [ ]:
df = pd.read_parquet(f'{DATA_DIR}/cleaned_telemetry.parquet')
# Ensure timestamp column exists (may be named differently in real data)
ts_col = 'Inicio' if 'Inicio' in df.columns else 'timestamp'
if ts_col in df.columns:
    df['timestamp'] = pd.to_datetime(df[ts_col])
print(f'Registros: {len(df):,}')
print(f'TAGs únicas: {df["TAG"].nunique()}')
print(f'Alarmes únicos: {df["Alarme"].nunique()}')

## 2. Distribuição de Alarmes por Criticidade

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribuição por Criticidade
crit_counts = df['Criticidade'].value_counts()
axes[0].bar(crit_counts.index, crit_counts.values)
axes[0].set_title('Alarmes por Criticidade')
axes[0].set_xlabel('Criticidade')
axes[0].set_ylabel('Contagem')
axes[0].tick_params(axis='x', rotation=20)

# Taxa de Don't Go por Tipo
dg_rate = df.groupby('Tipo')['Is_Dont_Go'].mean() * 100
axes[1].bar(dg_rate.index, dg_rate.values, color='tomato')
axes[1].set_title("Taxa de Don't Go por Tipo de Equipamento")
axes[1].set_ylabel("Don't Go (%)")

plt.tight_layout()
plt.savefig('../data/eda_criticidade_tipo.png', dpi=150)
plt.show()
print(f"Taxa global Don't Go: {df['Is_Dont_Go'].mean()*100:.4f}%")

## 3. Evolução Temporal dos Don't Go

In [ ]:
if 'timestamp' in df.columns:
    daily = df.set_index('timestamp').resample('D')['Is_Dont_Go'].sum()
    fig, ax = plt.subplots(figsize=(14, 4))
    daily.plot(ax=ax, color='steelblue')
    ax.set_title("Don't Go por Dia")
    ax.set_ylabel('Contagem')
    plt.tight_layout()
    plt.savefig('../data/eda_dontgo_timeline.png', dpi=150)
    plt.show()

## 4. Top Alarmes mais Frequentes

In [ ]:
top_alarmes = df['Alarme'].value_counts().head(20)
fig, ax = plt.subplots(figsize=(12, 5))
top_alarmes.plot(kind='bar', ax=ax)
ax.set_title('Top 20 Alarmes mais Frequentes')
ax.set_ylabel('Frequência')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../data/eda_top_alarmes.png', dpi=150)
plt.show()

## 5. Validação das Regras de Negócio

In [ ]:
rules = load_rules(f'{DATA_DIR}/Alarmes - Regra de Negocio.xlsx')
print(f'Regras carregadas: {len(rules)}')
print(list(rules.items())[:5])

In [ ]:
# Aplicar janela deslizante em amostra (full dataset é pesado)
sample = df.sample(n=min(50_000, len(df)), random_state=42)
if 'timestamp' in sample.columns:
    df_derived = apply_sliding_window(sample, rules)
    comparison = compare_signals(df_derived)
    print('Comparação sinal derivado vs registrado (amostra 50k):')
    for k, v in comparison.items():
        print(f'  {k}: {v}')
    df_derived[['TAG', 'Alarme', 'timestamp', 'Is_Dont_Go', 'Is_Dont_Go_derived']].to_csv(
        '../data/business_rules_validation.csv', index=False
    )
    print('Salvo em data/business_rules_validation.csv')
else:
    print('Coluna timestamp não encontrada. Verifique o dataset real.')